# Chatbot Data Preparation — Egyptian Real Estate

**Objective:** Prepare the cleaned property dataset for use by a Gemini-powered
conversational chatbot. The chatbot will use **function calling** (tools) over the
real data — no fine-tuning. This notebook parses complex columns (`images`,
`location`, `amenities`), creates derived features, and exports chatbot-ready
artifacts.

**Input:**
- `../../data/processed/cleaned_all_egypt.csv`
- `cluster` column from `../../data/processed/clustered_all_egypt.csv`

**Output** (all saved to `../../data/processed/chat_bot/`):
- `chatbot_properties.parquet` — final cleaned table with parsed lists
- `amenities_mapping.json` — code → readable English name
- `location_lookup.csv` — governorate / city / area with property counts

---
## 1. Setup & Imports

In [ ]:
# Standard library
import ast
import json
import os
import re
from collections import Counter
from pathlib import Path

# Data manipulation
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 25)
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.max_rows', 60)
sns.set_theme(style='whitegrid', palette='muted')

print('Libraries loaded successfully.')

In [ ]:
NOTEBOOK_DIR  = Path(os.getcwd())
PROJECT_ROOT  = NOTEBOOK_DIR.parents[1]   # notebooks/04_Chat_bot → project root

# Input paths
CLEANED_CSV   = PROJECT_ROOT / 'data' / 'processed' / 'cleaned_all_egypt.csv'
CLUSTERED_CSV = PROJECT_ROOT / 'data' / 'processed' / 'clustered_all_egypt.csv'

# Output directory (created if it does not exist)
OUTPUT_DIR    = PROJECT_ROOT / 'data' / 'processed' / 'chat_bot'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Project root : {PROJECT_ROOT}')
print(f'Cleaned CSV  : {CLEANED_CSV}')
print(f'Clustered CSV: {CLUSTERED_CSV}')
print(f'Output dir   : {OUTPUT_DIR}')

---
## 2. Load & Merge Data

Load the cleaned dataset and merge the `cluster` column from the clustered dataset
(join on `id`). The cluster column enables similar-property search in the chatbot.

In [ ]:
df_clean   = pd.read_csv(CLEANED_CSV)
df_cluster = pd.read_csv(CLUSTERED_CSV, usecols=['id', 'cluster'])

print(f'Cleaned  : {df_clean.shape}')
print(f'Clusters : {df_cluster.shape}')

# Merge cluster column
df = df_clean.merge(df_cluster, on='id', how='left')
print(f'Merged   : {df.shape}')
print(f'Cluster NaN: {df["cluster"].isna().sum()}')

In [ ]:
df.head(3)

In [ ]:
print(f'Columns ({len(df.columns)}):')
for i, col in enumerate(df.columns, 1):
    print(f'  {i:2d}. {col:20s}  dtype={str(df[col].dtype):8s}  nulls={df[col].isna().sum():,}')

---
## 3. Inspect Raw Formats (Before Parsing)

> **Principle:** Print and examine real values before writing any parsing logic.
> Do not assume formats.

### 3.1 Images Column

In [ ]:
print('dtype:', df['images'].dtype)
print('Nulls:', df['images'].isna().sum())
print()

# Show 5 raw cell values
for idx in df['images'].dropna().head(5).index:
    val = df.at[idx, 'images']
    preview = repr(val[:200]) + ('...' if len(str(val)) > 200 else '')
    print(f'Row {idx}: {preview}')
    print()

### 3.2 Location Column

In [ ]:
print('dtype:', df['location'].dtype)
print('Nulls:', df['location'].isna().sum())
print('Unique values:', df['location'].nunique())
print()

# Show 15 diverse values
print('Sample values (random_state=42):')
for val in df['location'].drop_duplicates().sample(15, random_state=42).values:
    parts = val.split(',')
    print(f'  ({len(parts)} parts)  "{val}"')

In [ ]:
# Distribution of comma-separated depth
depths = df['location'].dropna().apply(lambda x: len(x.split(',')))
print('Location depth (number of comma-separated parts):')
print(depths.value_counts().sort_index())

### 3.3 Amenities Column

In [ ]:
print('dtype:', df['amenities'].dtype)
print('Nulls:', df['amenities'].isna().sum())
print()

# Show 10 raw values
for idx in df['amenities'].dropna().head(10).index:
    print(f'Row {idx}: {repr(df.at[idx, "amenities"])}')

---
## 4. Parse & Clean Images

The `images` column is stored as a **stringified Python list** of URLs.
We parse it into a real list, remove empty strings and in-row duplicates,
and create an `images_count` feature.

In [ ]:
def parse_images(val):
    """Parse a stringified list of image URLs into a clean Python list.

    Handles NaN, malformed strings, empty strings inside the list,
    and duplicate URLs within a single property.
    """
    if pd.isna(val):
        return []
    try:
        imgs = ast.literal_eval(str(val))
        if not isinstance(imgs, list):
            return []
        # Strip whitespace, remove empty strings
        imgs = [url.strip() for url in imgs if isinstance(url, str) and url.strip()]
        # Remove duplicates while preserving order
        seen = set()
        unique = []
        for url in imgs:
            if url not in seen:
                seen.add(url)
                unique.append(url)
        return unique
    except (ValueError, SyntaxError):
        return []


# Apply
df['images_parsed'] = df['images'].apply(parse_images)
df['images_count']  = df['images_parsed'].apply(len)

print(f'Parsed images for {len(df):,} properties.')
print(f'Total image URLs: {df["images_count"].sum():,}')

In [ ]:
print('images_count distribution:')
print(df['images_count'].describe().to_string())
print()
print(f'Properties with 0 images: {(df["images_count"] == 0).sum():,} '
      f'({(df["images_count"] == 0).mean():.1%})')
print()

fig, ax = plt.subplots(figsize=(10, 4))
df['images_count'].clip(upper=20).value_counts().sort_index().plot.bar(
    ax=ax, color='steelblue', edgecolor='white')
ax.set_xlabel('Number of images (clipped at 20)')
ax.set_ylabel('Number of properties')
ax.set_title('Distribution of Image Count per Property')
plt.tight_layout()
plt.show()

---
## 5. Parse & Clean Location

Location strings are comma-separated from **most specific** to **least specific**.

Example: `"Villette, 5th Settlement Compounds, The 5th Settlement, New Cairo City, Cairo"`

**Extraction logic:**
| New column | Source |
|---|---|
| `governorate` | Last part (e.g., Cairo, Giza, Alexandria) |
| `city` | Second-to-last part (e.g., New Cairo City, Sheikh Zayed City) |
| `area` | First part — the most specific name (compound, street, neighborhood) |

In [ ]:
def parse_location(loc_str):
    """Extract governorate, city, and area from a hierarchical location string.

    Args:
        loc_str: Comma-separated string, most specific to least specific.

    Returns:
        dict with keys: governorate, city, area
    """
    if pd.isna(loc_str) or not str(loc_str).strip():
        return {'governorate': None, 'city': None, 'area': None}

    parts = [p.strip() for p in str(loc_str).split(',')]
    parts = [p for p in parts if p]  # drop empty segments

    if len(parts) >= 3:
        return {'governorate': parts[-1], 'city': parts[-2], 'area': parts[0]}
    elif len(parts) == 2:
        return {'governorate': parts[-1], 'city': parts[0], 'area': None}
    elif len(parts) == 1:
        return {'governorate': parts[0], 'city': None, 'area': None}
    return {'governorate': None, 'city': None, 'area': None}


# Apply
loc_df = df['location'].apply(parse_location).apply(pd.Series)
df[['governorate', 'city', 'area']] = loc_df

print('Location parsed. Preview:')
df[['location', 'governorate', 'city', 'area']].sample(8, random_state=42)

In [ ]:
def normalize_text(s):
    """Normalize a location string: strip edges, collapse whitespace."""
    if pd.isna(s):
        return s
    s = str(s).strip()
    s = re.sub(r'\s+', ' ', s)
    return s


for col in ['governorate', 'city', 'area']:
    df[col] = df[col].apply(normalize_text)

print('Whitespace normalization applied.')

In [ ]:
print(f'Unique governorates: {df["governorate"].nunique()}')
print()
print(df['governorate'].value_counts())

In [ ]:
print(f'Unique cities: {df["city"].nunique()}')
print()
print(df['city'].value_counts().head(25))

In [ ]:
print(f'Unique areas: {df["area"].nunique()}')
print(f'Null areas  : {df["area"].isna().sum():,}')
print()
print(df['area'].value_counts().head(25))

### 5.1 Flag Suspicious Location Values

Check for governorate values that don't match known Egyptian governorates,
and cities that appear under multiple governorates (possible parsing issues).

In [ ]:
KNOWN_GOVERNORATES = {
    'Cairo', 'Giza', 'Alexandria', 'Dakahlia', 'Red Sea', 'Beheira',
    'Fayoum', 'Gharbiya', 'Ismailia', 'Menofia', 'Minya', 'Qaliubiya',
    'New Valley', 'Suez', 'Aswan', 'Assiut', 'Beni Suef', 'Port Said',
    'Damietta', 'Sharkia', 'South Sinai', 'Kafr Al Sheikh', 'Matrouh',
    'Luxor', 'Qena', 'North Sinai', 'Sohag',
}

actual_govs = set(df['governorate'].dropna().unique())
unexpected = actual_govs - KNOWN_GOVERNORATES

if unexpected:
    print(f'\u26a0\ufe0f  {len(unexpected)} unexpected governorate value(s) — review these:')
    for g in sorted(unexpected):
        count = (df['governorate'] == g).sum()
        print(f'\n  \u2022 "{g}" \u2014 {count:,} properties')
        samples = df.loc[df['governorate'] == g, 'location'].head(3).tolist()
        for s in samples:
            print(f'      e.g. "{s}"')
else:
    print('\u2705 All governorate values match known Egyptian governorates.')

In [ ]:
# Cities appearing under multiple governorates (potential parsing issues)
city_gov = df.dropna(subset=['city']).groupby('city')['governorate'].nunique()
multi_gov = city_gov[city_gov > 1].sort_values(ascending=False)

if len(multi_gov) > 0:
    print(f'\u26a0\ufe0f  {len(multi_gov)} city name(s) appear under multiple governorates:')
    for city_name in multi_gov.head(15).index:
        govs = df.loc[df['city'] == city_name, 'governorate'].value_counts()
        print(f'\n  "{city_name}":')
        for g, c in govs.items():
            print(f'    \u2192 {g}: {c:,}')
else:
    print('\u2705 Each city maps to exactly one governorate.')

---
## 6. Parse & Clean Amenities

The `amenities` column stores **stringified Python lists** of 2-letter codes
(e.g., `"['AC', 'BW', 'CP']"` ).

**Steps:**
1. Parse into real Python lists
2. Collect all unique codes and their frequency
3. Build a code → readable-name mapping (uncertain codes are flagged)
4. Create `amenities_readable` and `amenities_count`

In [ ]:
def parse_amenities(val):
    """Parse a stringified list of amenity codes into a Python list."""
    if pd.isna(val):
        return []
    try:
        codes = ast.literal_eval(str(val))
        if isinstance(codes, list):
            return [str(c).strip() for c in codes if str(c).strip()]
        return []
    except (ValueError, SyntaxError):
        return []


df['amenities_parsed'] = df['amenities'].apply(parse_amenities)
df['amenities_count']  = df['amenities_parsed'].apply(len)

print(f'Parsed amenities for {len(df):,} properties.')
print(f'Properties with 0 amenities: {(df["amenities_count"] == 0).sum():,}')

In [ ]:
# All unique codes with their frequency
all_codes = [code for codes_list in df['amenities_parsed'] for code in codes_list]
code_freq = Counter(all_codes)

print(f'Total amenity mentions: {len(all_codes):,}')
print(f'Unique codes: {len(code_freq)}')
print()
print('Code frequencies (descending):')
for c, count in code_freq.most_common():
    pct = count / len(df) * 100
    print(f'  {c:4s} \u2192 {count:>6,}  ({pct:5.1f}% of properties)')

In [ ]:
# ── Amenity code mapping ──────────────────────────────────────────────
# Codes marked "UNKNOWN" need verification against propertyfinder.eg

AMENITY_MAPPING = {
    # Confident mappings
    'AC': 'Central A/C',
    'BA': 'Balcony',
    'BK': 'Built-in Kitchen Appliances',
    'BW': 'Built-in Wardrobes',
    'CP': 'Covered Parking',
    'MR': "Maid's Room",
    'PG': 'Private Garden',
    'PP': 'Private Pool',
    'SE': 'Security',
    'SP': 'Shared Pool',
    'VW': 'View',
    'WC': 'Walk-in Closet',
    # ── Needs verification against propertyfinder.eg ───────────
    'BL': 'UNKNOWN \u2014 verify (Basement?)',
    'CO': 'UNKNOWN \u2014 verify (Concierge?)',
    'LB': 'UNKNOWN \u2014 verify (Lobby in Building?)',
    'SS': 'UNKNOWN \u2014 verify (Shared Spa?)',
    'ST': 'UNKNOWN \u2014 verify (Study? Storage?)',
    'SY': 'UNKNOWN \u2014 verify (Shared Gym?)',
}

# Safety check: any codes in data not in our mapping?
data_codes   = set(code_freq.keys())
mapped_codes = set(AMENITY_MAPPING.keys())
unmapped     = data_codes - mapped_codes

if unmapped:
    print(f'\u26a0\ufe0f  Codes in data but NOT in mapping: {unmapped}')
    for c in sorted(unmapped):
        AMENITY_MAPPING[c] = f'UNKNOWN \u2014 verify ({c})'
        print(f'  Added placeholder for: {c}')
else:
    print('\u2705 All codes in data are covered by the mapping.')

unknown_n = sum(1 for v in AMENITY_MAPPING.values() if 'UNKNOWN' in v)
print(f'\nConfident mappings : {len(AMENITY_MAPPING) - unknown_n}')
print(f'Need verification  : {unknown_n}')

In [ ]:
def map_amenities(codes_list):
    """Convert a list of amenity codes to readable English names."""
    return [AMENITY_MAPPING.get(c, f'UNKNOWN ({c})') for c in codes_list]


df['amenities_readable'] = df['amenities_parsed'].apply(map_amenities)

# Show a few examples
print('Sample amenities (codes \u2192 readable names):')
for _, row in df[df['amenities_count'] > 3].sample(5, random_state=42).iterrows():
    print(f'  Codes   : {row["amenities_parsed"]}')
    print(f'  Readable: {row["amenities_readable"]}')
    print()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
df['amenities_count'].clip(upper=15).value_counts().sort_index().plot.bar(
    ax=ax, color='teal', edgecolor='white')
ax.set_xlabel('Number of amenities')
ax.set_ylabel('Number of properties')
ax.set_title('Distribution of Amenity Count per Property')
plt.tight_layout()
plt.show()

---
## 7. Sanity Checks

Quick validation: null counts, numeric ranges by category, and duplicate IDs.

In [ ]:
print('Null values per column:')
print()
nulls     = df.isnull().sum()
nulls_pct = (nulls / len(df) * 100).round(2)
null_df   = pd.DataFrame({'nulls': nulls, 'pct (%)': nulls_pct})
display(null_df[null_df['nulls'] > 0].sort_values('nulls', ascending=False))

if null_df['nulls'].sum() == 0:
    print('\n\u2705 No null values in any column.')

In [ ]:
print('Numeric ranges by category:')
for cat in sorted(df['category'].unique()):
    sub = df[df['category'] == cat]
    print(f'\n\u2500\u2500 {cat} ({len(sub):,} rows) \u2500\u2500')
    for col in ['price', 'size', 'bedrooms', 'bathrooms']:
        s = sub[col].dropna()
        if len(s) > 0:
            print(f'  {col:12s}  min={s.min():>14,.1f}  median={s.median():>14,.1f}  max={s.max():>14,.1f}')

In [ ]:
dup_ids = df['id'].duplicated().sum()
print(f'Duplicate IDs: {dup_ids}')
if dup_ids > 0:
    print('\u26a0\ufe0f  Duplicate IDs found — showing first 10:')
    dups = df[df['id'].duplicated(keep=False)].sort_values('id')
    display(dups[['id', 'title', 'category', 'price']].head(10))
else:
    print('\u2705 All IDs are unique.')

---
## 8. Save Outputs

All outputs are saved to `data/processed/chat_bot/`:

| File | Description |
|------|-------------|
| `chatbot_properties.parquet` | Full cleaned table — lists stored natively in Parquet |
| `amenities_mapping.json` | Code → readable English name dictionary |
| `location_lookup.csv` | Governorate / city / area with property counts |

In [ ]:
# Prepare final dataframe: replace stringified columns with parsed versions
df_final = df.copy()

# Overwrite the original string columns with parsed Python lists
df_final['images']    = df_final['images_parsed']
df_final['amenities'] = df_final['amenities_parsed']

# Drop intermediate columns
df_final.drop(columns=['images_parsed', 'amenities_parsed'], inplace=True)

print(f'Final dataframe: {df_final.shape[0]:,} rows \u00d7 {df_final.shape[1]} columns')
print()
print('Columns:')
for c in df_final.columns:
    print(f'  \u2022 {c}')

In [ ]:
# ── Save Parquet ──────────────────────────────────────────────────────
parquet_path = OUTPUT_DIR / 'chatbot_properties.parquet'
try:
    df_final.to_parquet(parquet_path, index=False, engine='pyarrow')
    print(f'\u2705 Saved: {parquet_path}')
    print(f'   Shape: {df_final.shape}')
except ImportError:
    print('\u274c pyarrow is not installed.')
    print('   Please run:  pip install pyarrow')
    print('   Then re-run this cell.')

In [ ]:
# ── Save amenities mapping ───────────────────────────────────────────
json_path = OUTPUT_DIR / 'amenities_mapping.json'
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(AMENITY_MAPPING, f, indent=2, ensure_ascii=False)

print(f'\u2705 Saved: {json_path}')
print(f'   {len(AMENITY_MAPPING)} code mappings')

In [ ]:
# ── Save location lookup ─────────────────────────────────────────────
location_lookup = (
    df_final
    .groupby(['governorate', 'city', 'area'], dropna=False)
    .size()
    .reset_index(name='num_properties')
    .sort_values('num_properties', ascending=False)
)

lookup_path = OUTPUT_DIR / 'location_lookup.csv'
location_lookup.to_csv(lookup_path, index=False)

print(f'\u2705 Saved: {lookup_path}')
print(f'   {len(location_lookup):,} unique location combinations')

In [ ]:
print('=' * 60)
print('SAVE SUMMARY')
print('=' * 60)
print(f'Output directory: {OUTPUT_DIR}')
print()
print(f'1. chatbot_properties.parquet  \u2014 {df_final.shape[0]:,} rows \u00d7 {df_final.shape[1]} cols')
print(f'2. amenities_mapping.json      \u2014 {len(AMENITY_MAPPING)} codes')
print(f'3. location_lookup.csv         \u2014 {len(location_lookup):,} location combos')
print('=' * 60)

---
## 9. Findings & Next Steps

### Decisions needed (review before proceeding)

1. **Unknown amenity codes** — The following codes need verification against
   [propertyfinder.eg](https://www.propertyfinder.eg). Update `AMENITY_MAPPING`
   above and re-export `amenities_mapping.json`:
   - `BL` — Basement? Laundry?
   - `CO` — Concierge? Compound? Community?
   - `LB` — Lobby in Building?
   - `SS` — Shared Spa? Sauna?
   - `ST` — Study? Storage?
   - `SY` — Shared Gym? Security System? CCTV?

2. **Suspicious locations** — Review the flagged governorate and city values above.
   Some location strings may have fewer or more parts than expected, causing
   mis-classification.

3. **Missing values:**
   - `bedrooms` has ~10k nulls (mostly commercial properties — expected)
   - `bathrooms` has ~2.7k nulls
   - `area` (parsed) will be null for 2-part location strings (only governorate + city)
   - **Decision:** should the chatbot show these as "Not specified" or filter them?

4. **Image deduplication** — Many listings carry the same image in two resolutions
   (416×272 and 668×452). Should the chatbot keep only the larger version to reduce
   payload size?

### Planned notebooks

| # | Notebook | Purpose |
|---|----------|---------|
| 02 | `02_Chatbot_Tools.ipynb` | Define Python tool functions (search, filter, count, recommend) |
| 03 | `03_Gemini_Function_Calling.ipynb` | Wire tools to Gemini function-calling API |
| 04 | `04_Chatbot_Evaluation.ipynb` | Test with sample queries, measure accuracy & latency |